# Drive-only source ingestion

This notebook downloads the official dunnhumby Complete Journey package directly into the Google Drive project folder, extracts the eight CSVs into `01_raw_source`, and prints provenance metadata. It does not write raw or curated data to the Colab runtime or GitHub.

Before running: confirm the applicable source-use/redistribution terms and change `PROJECT_ROOT` only if the Drive folder has a different path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, shutil, urllib.request, zipfile

PROJECT_ROOT = Path('/content/drive/MyDrive/Retail DA - Customer & Commercial Intelligence')
RAW_DIR = PROJECT_ROOT / '01_raw_source'
DOCS_DIR = PROJECT_ROOT / '06_source_docs'
ZIP_PATH = DOCS_DIR / 'dunnhumby_The-Complete-Journey.zip'
DIRECT_ZIP = 'https://downloads.ctfassets.net/psj0p18eh7z1/3e9OAF7F9ONT4pwJc1luEw/d56af8aabad51bdb9888aad0240bd105/dunnhumby_The-Complete-Journey.zip'
EXPECTED = ['transaction_data.csv', 'causal_data.csv', 'coupon.csv', 'coupon_redempt.csv', 'campaign_table.csv', 'campaign_desc.csv', 'product.csv', 'hh_demographic.csv']

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Drive project folder not found: {PROJECT_ROOT}')
RAW_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)
print('Drive project:', PROJECT_ROOT)
print('Raw target:', RAW_DIR)
print('Package target:', ZIP_PATH)

In [ ]:
# Stream the package straight to Drive; no raw source is saved in /content.
with urllib.request.urlopen(DIRECT_ZIP) as response, open(ZIP_PATH, 'wb') as target:
    total = int(response.headers.get('Content-Length', '0'))
    written = 0
    while True:
        chunk = response.read(8 * 1024 * 1024)
        if not chunk:
            break
        target.write(chunk)
        written += len(chunk)
        print(f'\rDownloaded {written / 1024 / 1024:.1f} MB / {total / 1024 / 1024:.1f} MB', end='')
print('\nSaved on Drive:', ZIP_PATH, ZIP_PATH.stat().st_size, 'bytes')

In [ ]:
# Extract only safe paths, then flatten the eight expected CSVs into 01_raw_source.
with zipfile.ZipFile(ZIP_PATH) as archive:
    for member in archive.infolist():
        candidate = (RAW_DIR / member.filename).resolve()
        if RAW_DIR.resolve() not in candidate.parents and candidate != RAW_DIR.resolve():
            raise RuntimeError(f'Unsafe archive path: {member.filename}')
    archive.extractall(RAW_DIR)

for name in EXPECTED:
    matches = list(RAW_DIR.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {name}, found {len(matches)}')
    source = matches[0]
    target = RAW_DIR / name
    if source != target:
        if target.exists():
            raise FileExistsError(target)
        shutil.move(str(source), str(target))

unexpected = [p.name for p in RAW_DIR.iterdir() if p.is_file() and p.name not in EXPECTED]
if unexpected:
    print('Review non-source files in 01_raw_source:', unexpected)
print('Extracted expected files:', EXPECTED)

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

provenance = []
for name in EXPECTED:
    path = RAW_DIR / name
    provenance.append({
        'file_name': name,
        'drive_path': str(path),
        'acquired_at_utc': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(),
        'source_url': DIRECT_ZIP,
        'version': 'The Complete Journey package; verify source page terms',
        'sha256': sha256(path),
        'size_bytes': path.stat().st_size,
    })
PROVENANCE_PATH = DOCS_DIR / 'source_provenance.json'
PROVENANCE_PATH.write_text(json.dumps(provenance, indent=2), encoding='utf-8')
print(json.dumps(provenance, indent=2))
print('Provenance saved on Drive:', PROVENANCE_PATH)
print('Next: copy URL/date/version/SHA-256 into the Drive Source Register, then run the Drive-only pipeline from the repository.')